# Лабораторная работа 5. Градиентный бустинг

    ## Данные и протокол

    - **Классификация.** [Optical Recognition of Handwritten Digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), 1797 изображений размером 8×8, классы от 0 до 9. Практическая задача — распознавание цифр в оцифрованных рукописных формах. Используется поставляемая в sklearn часть UCI Optdigits.
    - **Регрессия.** [Diabetes (Efron и соавт.)](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html), 442 наблюдения и 10 исходных признаков. Цель — численный показатель прогрессирования болезни через год после исходного обследования. Это учебный эксперимент на исторической выборке.

    Для классификации основная метрика **macro F1**, она одинаково взвешивает каждый класс. Accuracy показывает долю верных ответов. Для регрессии основная метрика **MAE** выражает типичную абсолютную ошибку в единицах цели. RMSE сильнее реагирует на крупные промахи. R² показывает качество относительно прогноза средним.

    Разбиение 80/20 с seed 42 одинаково в пяти работах. Для классификации используется стратификация. Гиперпараметры выбираем по среднему результату 5-fold CV на обучающей части. Масштабирование, где оно используется, обучаем заново внутри каждого фолда и на полной обучающей выборке перед оценкой на тесте.

In [1]:
# Импортируем зависимости, задаём разбиение и функции расчёта метрик.
from hashlib import sha256
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp, softmax
from sklearn.datasets import load_diabetes, load_digits
from sklearn.ensemble import (GradientBoostingClassifier, GradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from threadpoolctl import threadpool_limits

SEED = 42

def load_task(classification):
    data = load_digits() if classification else load_diabetes(scaled=False)
    ids = np.arange(len(data.target))
    train, test = train_test_split(ids, test_size=0.2, random_state=SEED,
                                   stratify=data.target if classification else None)
    return data, train, test

def partition(data, train, test, scale):
    X_train, X_test = data.data[train], data.data[test]
    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
    return X_train, X_test

def metrics(y, prediction, classification):
    if classification:
        return {"macro_f1": f1_score(y, prediction, average="macro", zero_division=0),
                "accuracy": accuracy_score(y, prediction)}
    return {"mae": mean_absolute_error(y, prediction),
            "rmse": mean_squared_error(y, prediction) ** 0.5,
            "r2": r2_score(y, prediction)}

In [2]:
# Проверяем размеры и распределение обучающих данных.
for classification in (True, False):
    data, train, test = load_task(classification)
    print("Классификация" if classification else "Регрессия")
    print(f"Обучение: {len(train)}, тест: {len(test)}, признаков: {data.data.shape[1]}")
    print("Пропуски в обучении:", np.isnan(data.data[train]).sum())
    print(np.bincount(data.target[train]) if classification
          else pd.Series(data.target[train]).describe().round(2).to_string())

Классификация
Обучение: 1437, тест: 360, признаков: 64
Пропуски в обучении: 0
[142 146 142 146 145 145 145 143 139 144]
Регрессия
Обучение: 353, тест: 89, признаков: 10
Пропуски в обучении: 0
count    353.00
mean     153.74
std       78.06
min       25.00
25%       86.00
50%      142.00
75%      214.00
max      346.00


## Бейзлайн sklearn

Фиксируем исходную конфигурацию и измеряем её качество на отложенной части.

In [3]:
# Задаём исходные параметры sklearn и функцию оценки модели.
LAB = 5
def make_model(classification, own, params):
    params = params.copy()
    params.pop("scale")
    if LAB == 1:
        if own:
            return OwnKNN(classification, **params)
        return (KNeighborsClassifier if classification else KNeighborsRegressor)(**params)
    if LAB == 2:
        strength = params["strength"]
        if own:
            return OwnLinear(classification, strength)
        if classification:
            return LogisticRegression(C=strength, max_iter=2500, tol=1e-7)
        return Ridge(alpha=strength) if strength else LinearRegression()
    if LAB == 3:
        if own:
            return OwnTree(classification, **params)
        return (DecisionTreeClassifier if classification else DecisionTreeRegressor)(
            random_state=SEED, **params)
    if LAB == 4:
        if own:
            return OwnForest(classification, **params)
        return (RandomForestClassifier if classification else RandomForestRegressor)(
            random_state=SEED, n_jobs=1, **params)
    if own:
        return OwnBoost(classification, **params)
    return (GradientBoostingClassifier if classification else GradientBoostingRegressor)(
        random_state=SEED, **params)

def evaluate(data, train, test, classification, params, own=False):
    X_train, X_test = partition(data, train, test, params["scale"])
    model = make_model(classification, own, params)
    with threadpool_limits(limits=1):
        model.fit(X_train, data.target[train])
        prediction = model.predict(X_test)
    return metrics(data.target[test], prediction, classification)

def validate_candidates(data, train, classification, candidates):
    splitter = (StratifiedKFold if classification else KFold)(
        n_splits=5, shuffle=True, random_state=SEED)
    folds = list(splitter.split(train, data.target[train]))
    metric = "macro_f1" if classification else "mae"
    rows = []
    for hypothesis, params in candidates:
        scores = [evaluate(data, train[fit], train[val], classification, params)[metric]
                  for fit, val in folds]
        rows.append({"гипотеза": hypothesis, "mean": np.mean(scores), "std": np.std(scores)})
    table = pd.DataFrame(rows).set_index("гипотеза")
    winner = table["mean"].idxmax() if classification else table["mean"].idxmin()
    return table, winner, dict(candidates)[winner]

In [4]:
# Обучаем базовые библиотечные модели и считаем метрики.
bases = {}
baseline_scores = {}
for classification in (True, False):
    params = {'scale': False, **{'n_estimators': 30, 'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 1, 'subsample': 1.0}}
    if LAB == 2:
        params['strength'] = 1.0 if classification else 0.0
    if LAB == 4:
        params['max_features'] = 'sqrt' if classification else None
    bases[classification] = params
    data, train, test = load_task(classification)
    baseline_scores[classification] = evaluate(data, train, test, classification, params)
    print(('Классификация' if classification else 'Регрессия'), baseline_scores[classification])

Классификация {'macro_f1': 0.9324864381312736, 'accuracy': 0.9333333333333333}
Регрессия {'mae': 43.56105054147046, 'rmse': 53.05719230724078, 'r2': 0.4686702478244976}


## Проверка гипотез и улучшенный бейзлайн

Меньшая глубина, больший размер листа, уменьшенный шаг и подвыборка могут повысить устойчивость бустинга. Число итераций меняем вместе с шагом или глубиной в соответствующих конфигурациях.

Исходная конфигурация тоже участвует в CV. Если улучшения не подтверждаются, сохраняем исходные параметры.

In [5]:
# Сравниваем конфигурации на обучающих фолдах, затем оцениваем победителя на тесте.
chosen = {}
improved_scores = {}
for classification in (True, False):
    base = bases[classification]
    changes = [('Мелкие деревья и больше итераций', {'max_depth': 2, 'n_estimators': 60}), ('Меньший шаг с подвыборкой', {'learning_rate': 0.06, 'n_estimators': 60, 'subsample': 0.8}), ('Листья от 5 объектов и глубина до 2', {'max_depth': 2, 'min_samples_leaf': 5})]
    if LAB == 2:
        strengths = [0.01, 0.1, 10.0] if classification else [0.1, 1.0, 10.0, 50.0]
        changes = [('Масштабирование', {'scale': True})]
        changes += [(f'Масштабирование и штраф {v}', {'scale': True, 'strength': v})
                    for v in strengths]
    if LAB == 4:
        alternative = None if classification else 'sqrt'
        changes = [*changes, ('Другой размер подмножества признаков', {'max_features': alternative})]
    candidates = [('Бейзлайн', base)] + [(name, base | patch) for name, patch in changes]
    data, train, test = load_task(classification)
    table, winner, chosen[classification] = validate_candidates(data, train, classification, candidates)
    improved_scores[classification] = evaluate(data, train, test, classification, chosen[classification])
    print('Классификация' if classification else 'Регрессия')
    print(table.round(4).to_string())
    print('Выбор по CV:', winner, chosen[classification])
    print('Тест:', improved_scores[classification])

Классификация
                                       mean     std
гипотеза                                           
Бейзлайн                             0.9313  0.0161
Мелкие деревья и больше итераций     0.9462  0.0171
Меньший шаг с подвыборкой            0.9480  0.0122
Листья от 5 объектов и глубина до 2  0.9197  0.0244
Выбор по CV: Меньший шаг с подвыборкой {'scale': False, 'n_estimators': 60, 'learning_rate': 0.06, 'max_depth': 3, 'min_samples_leaf': 1, 'subsample': 0.8}
Тест: {'macro_f1': 0.9407536680584254, 'accuracy': 0.9416666666666667}
Регрессия
                                        mean     std
гипотеза                                            
Бейзлайн                             49.1907  3.3486
Мелкие деревья и больше итераций     48.2084  4.2333
Меньший шаг с подвыборкой            48.3473  2.8966
Листья от 5 объектов и глубина до 2  48.9291  3.8625
Выбор по CV: Мелкие деревья и больше итераций {'scale': False, 'n_estimators': 60, 'learning_rate': 0.1, 'max_depth': 2

## Собственная реализация

Для квадратичной ошибки строим дерево на остатках. Для классификации на каждом шаге строим по дереву на разности между индикатором класса и softmax-вероятностью. Собственная версия обновляет логиты шагом первого порядка, тогда как sklearn дополнительно корректирует значения в листьях. Одинаковые гиперпараметры поэтому дают разные классификаторы.

In [6]:
# Реализуем модель и предсказание средствами NumPy и SciPy.
class OwnTree:
    def __init__(self, classification, max_depth=None, min_samples_leaf=1,
                 max_features=None, random_state=42):
        self.classification = classification
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state

    def fit(self, X, y, classes=None):
        self.rng = np.random.default_rng(self.random_state)
        X = np.asarray(X, dtype=float)
        if self.classification:
            self.classes = np.unique(y) if classes is None else classes
            y = np.eye(len(self.classes))[np.searchsorted(self.classes, y)]
        else:
            y = np.asarray(y, dtype=float)[:, None]
        self.root = self._grow(X, y, depth=0)
        return self

    def _grow(self, X, y, depth):
        n, p = X.shape
        leaf = y.mean(axis=0)
        min_leaf = self.min_samples_leaf
        at_depth_limit = self.max_depth is not None and depth >= self.max_depth
        if n < 2 * min_leaf or at_depth_limit or np.all(y == y[0]):
            return leaf

        # Для обоих критериев максимизируем сумму квадратов сумм / размер дочернего узла.
        # Это эквивалентно минимизации взвешенной Gini или суммы квадратов ошибок.
        total = y.sum(axis=0)
        best_score = total @ total / n + 1e-10
        best = None
        if self.max_features == "sqrt":
            columns = self.rng.choice(p, max(1, int(np.sqrt(p))), replace=False)
        else:
            columns = range(p)
        for col in columns:
            order = np.argsort(X[:, col], kind="stable")
            values = X[order, col]
            cuts = np.flatnonzero(values[:-1] < values[1:]) + 1
            cuts = cuts[(cuts >= min_leaf) & (cuts <= n - min_leaf)]
            if not len(cuts):
                continue
            left = np.cumsum(y[order], axis=0)[cuts - 1]
            right = total - left
            scores = (left ** 2).sum(axis=1) / cuts
            scores += (right ** 2).sum(axis=1) / (n - cuts)
            index = scores.argmax()
            if scores[index] > best_score:
                best_score = scores[index]
                cut = cuts[index]
                best = col, (values[cut - 1] + values[cut]) / 2
        if best is None:
            return leaf

        col, threshold = best
        left_mask = X[:, col] <= threshold
        return (col, threshold, self._grow(X[left_mask], y[left_mask], depth + 1),
                self._grow(X[~left_mask], y[~left_mask], depth + 1))

    def leaf_values(self, X):
        predictions = []
        for row in X:
            node = self.root
            while isinstance(node, tuple):
                col, threshold, left, right = node
                node = left if row[col] <= threshold else right
            predictions.append(node)
        return np.asarray(predictions)

    def predict(self, X):
        values = self.leaf_values(X)
        return self.classes[values.argmax(axis=1)] if self.classification else values[:, 0]

In [7]:
# Реализуем модель и предсказание средствами NumPy и SciPy.
class OwnBoost:
    def __init__(self, classification, n_estimators=30, learning_rate=0.1,
                 max_depth=3, min_samples_leaf=1, subsample=1.0, random_state=42):
        self.classification = classification
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.subsample = subsample
        self.random_state = random_state

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        if self.classification:
            self.classes, encoded = np.unique(y, return_inverse=True)
            target = np.eye(len(self.classes))[encoded]
            self.initial = np.log(target.mean(axis=0))
        else:
            target = y[:, None]
            self.initial = target.mean(axis=0)
        scores = np.tile(self.initial, (len(X), 1))
        self.stages = []
        for _ in range(self.n_estimators):
            prediction = softmax(scores, axis=1) if self.classification else scores
            residual = target - prediction
            sample = (np.arange(len(X)) if self.subsample == 1 else
                      rng.choice(len(X), int(self.subsample * len(X)), replace=False))
            stage = []
            for output in range(target.shape[1]):
                tree = OwnTree(False, self.max_depth, self.min_samples_leaf)
                tree.fit(X[sample], residual[sample, output])
                scores[:, output] += self.learning_rate * tree.predict(X)
                stage.append(tree)
            self.stages.append(stage)
        return self

    def predict(self, X):
        scores = np.tile(self.initial, (len(X), 1))
        for stage in self.stages:
            for output, tree in enumerate(stage):
                scores[:, output] += self.learning_rate * tree.predict(X)
        return self.classes[scores.argmax(axis=1)] if self.classification else scores[:, 0]

In [8]:
# Сравниваем четыре варианта на одном тесте и сохраняем метрики для итогового ноутбука.
records = []
for classification in (True, False):
    data, train, test = load_task(classification)
    own_base = evaluate(data, train, test, classification, bases[classification], own=True)
    own_improved = evaluate(data, train, test, classification, chosen[classification], own=True)
    scores = {
        "sklearn_base": baseline_scores[classification],
        "sklearn_tuned": improved_scores[classification],
        "own_base": own_base,
        "own_tuned": own_improved,
    }
    task = "classification" if classification else "regression"
    print(task, pd.DataFrame(scores).T.round(4).to_string(), sep="\n")
    split = sha256(np.asarray(test, dtype="<i8").tobytes()).hexdigest()[:16]
    for version, values in scores.items():
        records.append(dict(lab=LAB, task=task, version=version, n_test=len(test), split=split, **values))

result_path = Path("results") / f"lab-{LAB}.json"
result_path.parent.mkdir(exist_ok=True)
result_path.write_text(json.dumps(records, indent=2) + "\n", encoding="utf-8")

classification
               macro_f1  accuracy
sklearn_base     0.9325    0.9333
sklearn_tuned    0.9408    0.9417
own_base         0.8460    0.8472
own_tuned        0.8802    0.8806
regression
                   mae     rmse      r2
sklearn_base   43.5611  53.0572  0.4687
sklearn_tuned  42.3288  52.0720  0.4882
own_base       43.5611  53.0572  0.4687
own_tuned      42.3288  52.0720  0.4882


## Выводы

**Классификация.** MACRO_F1 библиотечной модели улучшилось на тесте (0.9325 → 0.9408). Собственная реализация дала 0.8460 с базовыми параметрами и 0.8802 с настройками, выбранными по CV для sklearn. Разность между собственным и библиотечным вариантами после настройки составила -0.0605.

**Регрессия.** MAE библиотечной модели улучшилось на тесте (43.5611 → 42.3288). Собственная реализация дала 43.5611 с базовыми параметрами и 42.3288 с настройками, выбранными по CV для sklearn. Разность между собственным и библиотечным вариантами после настройки составила +0.0000.

Результаты относятся к одному отложенному разбиению. Выбранные по CV настройки не гарантируют улучшения на тесте.